## Homework 3

> [!NOTE]
> This homework uses the pinned 2026 lead-scoring release in the course
> repository. The plan and report are available in `cohorts/2026/data/`.

### Dataset

In [101]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [102]:
data = "https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv"

In [103]:
!wget $data -O course_lead_scoring_2026.csv

--2026-10-12 22:10:14--  https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv


Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.110.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 278746 (272K) [text/plain]
Saving to: ‘course_lead_scoring_2026.csv’

course_lead_scoring 100%[===================>] 272.21K  --.-KB/s    in 0.05s   

2026-10-12 22:10:14 (5.11 MB/s) - ‘course_lead_scoring_2026.csv’ saved [278746/278746]



In [104]:
df = pd.read_csv("course_lead_scoring_2026.csv")
df.head().T

,0,1,2,3,4
lead_source,organic_search,social_media,referral,paid_ads,referral
industry,technology,technology,retail,manufacturing,manufacturing
employment_status,employed,employed,employed,student,student
location,europe,south_america,europe,NaN,north_america
annual_income,88160.0,72688.0,44697.0,NaN,24062.0
number_of_courses_viewed,3,3,3,3,4
interaction_count,4,7,4,6,5
lead_score,0.64,0.72,0.58,0.57,0.62
converted,1,1,1,0,1


### Data preparation

In [105]:
df.columns = df.columns.str.lower().str.replace(" ", "_")
categorical_columns = list(df.select_dtypes(include="object").columns)
numerical_columns = list(df.select_dtypes(include="number").columns)


/tmp/ipykernel_954398/1650861609.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_columns = list(df.select_dtypes(include="object").columns)


In [106]:
df[categorical_columns].isnull().sum()

lead_source          148
industry             240
employment_status    194
location             208
dtype: int64

In [107]:
for col in categorical_columns:
    df[col] = df[col].fillna("NA")
for col in numerical_columns:
    df[col] = df[col].fillna(0.0)

In [108]:
df[categorical_columns].isnull().sum()

lead_source          0
industry             0
employment_status    0
location             0
dtype: int64

In [109]:
df[numerical_columns].isnull().sum()

annual_income               0
number_of_courses_viewed    0
interaction_count           0
lead_score                  0
converted                   0
dtype: int64

In [110]:
df['industry'].mode()

0    technology
Name: industry, dtype: str

## Question 1

What is the most frequent observation (mode) for the column `industry`?

- `NA`
- `technology`
- `healthcare`
- `retail`

Response = `technology`

## Question 2

Create the [correlation matrix](https://www.google.com/search?q=correlation+matrix) for the numerical features of your dataset.
In a correlation matrix, you compute the correlation coefficient between every pair of features.

What are the two features that have the biggest correlation?

- `interaction_count` and `lead_score`
- `number_of_courses_viewed` and `lead_score`
- `number_of_courses_viewed` and `interaction_count`
- `annual_income` and `interaction_count`

Only consider the pairs above when answering this question.

In [111]:
r1 = df.interaction_count.corr(df.lead_score)
float(r1)

0.9157457980418677

In [112]:
r2 = df.number_of_courses_viewed.corr(df.lead_score)
float(r2)   

0.7572042392104887

In [113]:
r3 = df.number_of_courses_viewed.corr(df.interaction_count)
float(r3)       

0.7216090038937009

In [114]:
r4 = df.annual_income.corr(df.interaction_count)
float(r4)           

0.12284235135955954

In [115]:
r1 > r2 and r1 > r3 and r1 > r4

np.True_


`interaction_count` and `lead_score` have the bigest correlation

## Split the data

- Split the data with these exact calls:

```python
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)
```

- Make sure that the target value `converted` is not in your dataframe.

In [116]:
from sklearn.model_selection import train_test_split

In [117]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)

In [118]:
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42)

In [119]:
len(df_train), len(df_val), len(df_test)

(3000, 1000, 1000)

### Question 3

- Calculate the mutual information score between `converted` and other categorical variables in the dataset. Use the training set only.
- Round the scores to 2 decimals using `round(score, 2)`.

Which of these variables has the biggest mutual information score?

- `industry`
- `location`
- `lead_source`
- `employment_status`

In [120]:
from sklearn.metrics import mutual_info_score

In [121]:
score_1 = mutual_info_score(df_full_train.converted, df_full_train.industry)
round(score_1, 2)

0.0

In [122]:
score_2 = mutual_info_score(df_full_train.converted, df_full_train.location)
round(score_2, 2)

0.0

In [123]:
score_3 = mutual_info_score(df_full_train.converted, df_full_train.lead_source)
round(score_3, 2)   

0.03

In [124]:
score_4 = mutual_info_score(df_full_train.converted, df_full_train.employment_status)
round(score_4, 2)   

0.02

lead_source has the biggest mutual information score with the target variable 'converted', which means it is the most informative feature among the categorical features we have analyzed. 

### Question 4

- Now let's train a logistic regression.
- Remember that we have several categorical variables in the dataset. Include them using one-hot encoding.
- Fit the model on the training dataset.
  - To make sure the results are reproducible across different versions of Scikit-Learn, fit the model with these parameters:
  - `model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)`
- Calculate the accuracy on the validation dataset and round it to 2 decimal digits.

What accuracy did you get?

- 0.55
- 0.65
- 0.75
- 0.85

In [125]:
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction import DictVectorizer

In [126]:
y_train = df_train['converted']
y_val = df_val['converted']
y_test = df_test['converted']

In [127]:
del df_train['converted']
del df_val['converted']
del df_test['converted']

In [128]:
categorical_columns = list(df_train.select_dtypes(include="object").columns)
numerical_columns   = list(df_train.select_dtypes(include="number").columns)
feature_columns = categorical_columns + numerical_columns
train_dict = df_train[feature_columns].to_dict(orient='records')
val_dict = df_val[feature_columns].to_dict(orient='records')
test_dict = df_test[feature_columns].to_dict(orient='records')
dv = DictVectorizer(sparse=False)
dv.fit(train_dict)

/tmp/ipykernel_954398/999547218.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_columns = list(df_train.select_dtypes(include="object").columns)


DictVectorizer(sparse=False)

In [129]:
X_train = dv.transform(train_dict)
model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train, y_train)

,"random_state random_state: int, RandomState instance, default=NoneOnly used for `solver` == 'sag', 'saga' or 'liblinear' to shuffle thedata. It has no effect on the other solvers.See :term:`Glossary <random_state>` for details.",42
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`multiclass` problems (`n_classes >= 3`), all solvers except 'liblinear' minimize the full multinomial loss, 'liblinear' will raise an error.- 'newton-cholesky' is a good choice for `n_samples` >> `n_features * n_classes`, especially with one-hot encoded categorical features with rare categories. Be aware that the memory usage of this solver has a quadratic dependency on `n_features * n_classes` because it explicitly computes the full Hessian matrix.- For small datasets, 'liblinear' is a good choice, whereas 'sag' and 'saga' are faster for large ones;- 'liblinear' can only handle binary classification by default. To apply a one-versus-rest scheme for the multiclass setting one can wrap it with the :class:`~sklearn.multiclass.OneVsRestClassifier`... warning:: The choice of the algorithm depends on the penalty chosen (`l1_ratio=0` for L2-penalty, `l1_ratio=1` for L1-penalty and `0 < l1_ratio < 1` for Elastic-Net) and on (multinomial) multiclass support: ================= ======================== ====================== solver l1_ratio multinomial multiclass ================= ======================== ====================== 'lbfgs' l1_ratio=0 yes 'liblinear' l1_ratio=1 or l1_ratio=0 no 'newton-cg' l1_ratio=0 yes 'newton-cholesky' l1_ratio=0 yes 'sag' l1_ratio=0 yes 'saga' 0<=l1_ratio<=1 yes ================= ======================== ======================.. note:: 'sag' and 'saga' fast convergence is only guaranteed on features with approximately the same scale. You can preprocess the data with a scaler from :mod:`sklearn.preprocessing`... seealso:: Refer to the :ref:`User Guide <Logistic_regression>` for more information regarding :class:`LogisticRegression` and more specifically the :ref:`Table <logistic_regression_solvers>` summarizing solver/penalty supports... versionadded:: 0.17 Stochastic Average Gradient (SAG) descent solver. Multinomial support in version 0.18... versionadded:: 0.19 SAGA solver... versionchanged:: 0.22 The default solver changed from 'liblinear' to 'lbfgs' in 0.22... versionadded:: 1.2 newton-cholesky solver. Multinomial support in version 1.6.",'liblinear'
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l

In [130]:
from sklearn.metrics import accuracy_score

In [131]:
y_pred = model.predict(dv.transform(val_dict))
accuracy_score(y_val, y_pred)
round(accuracy_score(y_val, y_pred), 2)

0.65

The accuracy on the validation dataset is 0.65

### Question 5

- Let's find the least useful feature using the _feature elimination_ technique.
- Train a model using the same features and parameters as in Q4 (without rounding).
- Now exclude each feature from this set and train a model without it. Record the accuracy for each model.
- For each feature, calculate the difference between the original accuracy and the accuracy without the feature.

Which of following feature has the smallest difference?

- `'lead_source'`
- `'number_of_courses_viewed'`
- `'interaction_count'`

In [132]:
y_pred = model.predict(dv.transform(val_dict))
accuracy_score(y_val, y_pred)

0.645

In [133]:

feature_names = list(feature_columns)
train_dict = df_train[feature_names].to_dict(orient='records')
val_dict = df_val[feature_names].to_dict(orient='records')
X_train_base = dv.transform(train_dict)
model_base = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model_base.fit(X_train_base, y_train)
y_pred = model_base.predict(dv.transform(val_dict))
accuracy_base = accuracy_score(y_val, y_pred)


for col in feature_names:
    reduced = [c for c in feature_names if c != col]

    train_reduced = df_train[reduced].to_dict(orient='records')
    val_reduced = df_val[reduced].to_dict(orient='records')

    X_train_reduced = dv.transform(train_reduced)
    X_val_reduced = dv.transform(val_reduced)
    m = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
    m.fit(X_train_reduced, y_train)
    y_pred = m.predict(X_val_reduced)
    accuracy = accuracy_score(y_val, y_pred)
    diff = accuracy_base - accuracy
        
    print(f"Accuracy without {col}: {accuracy} (Difference: {diff})")
        
    

Accuracy without lead_source: 0.642 (Difference: 0.0030000000000000027)
Accuracy without industry: 0.645 (Difference: 0.0)
Accuracy without employment_status: 0.645 (Difference: 0.0)
Accuracy without location: 0.645 (Difference: 0.0)
Accuracy without annual_income: 0.724 (Difference: -0.07899999999999996)
Accuracy without number_of_courses_viewed: 0.643 (Difference: 0.0020000000000000018)
Accuracy without interaction_count: 0.601 (Difference: 0.04400000000000004)
Accuracy without lead_score: 0.644 (Difference: 0.0010000000000000009)


response : number_of_courses_viewed

### Question 6

- Now let's train a regularized logistic regression.
- Let's try the following values of the parameter `C`: `[0.000001, 0.00001, 0.0001, 0.001]`.
- Train models using all the features as in Q4.
- Calculate the accuracy on the validation dataset and round it to 3 decimal digits.

Which of these `C` leads to the best accuracy on the validation set?

- 0.000001
- 0.00001
- 0.0001
- 0.001

In [134]:
for C in [0.000001, 0.00001, 0.0001, 0.001]:
        
        train_regularized = df_train[reduced].to_dict(orient='records')
        val_regularized = df_val[reduced].to_dict(orient='records')

        X_train_regularized = dv.transform(train_regularized)
        X_val_regularized = dv.transform(val_regularized)
        m = LogisticRegression(solver='liblinear', C=C, max_iter=1000, random_state=42)
        m.fit(X_train_regularized, y_train)
        y_pred = m.predict(X_val_regularized)
        accuracy = accuracy_score(y_val, y_pred)

        print(f"Accuracy with C={C}: {round(accuracy, 2)}")

Accuracy with C=1e-06: 0.6
Accuracy with C=1e-05: 0.6
Accuracy with C=0.0001: 0.61
Accuracy with C=0.001: 0.64


Accuracy with C=0.001: 0.64 is the best 